In [ ]:
from optics_design_workbench.jupyter_utils import *
from matplotlib.pyplot import *
import IPython.display
import threading
import time
import psutil

# start endless simulation in background thread

In [ ]:
t0 = time.time()
def runEndlessSimulation():
  try:
    with FreecadDocument('main.FCStd', showProgress=False) as f:
      f.runSimulation('true', endIf=lambda r: time.time()-t0>14.1*60*60)
  # retry once if 'ungently exit' error is raised because this simulation is always exited ungently by this test notebook
  except RuntimeError as e:
    if 'ungently' in str(e):
      time.sleep(30)
      with FreecadDocument('main.FCStd', showProgress=False) as f:
        f.runSimulation('true', endIf=lambda r: time.time()-t0>14.1*60*60)
    else:
      raise

thread = threading.Thread(target=runEndlessSimulation, daemon=True)
thread.start()  

In [ ]:
time.sleep(120)

# record memory consumption for 14 hours

Fourteen hours make sure a worker endOfLife is experienced (happens randomly between 10 and 12 hours)

Track memory consumption to make sure slope of memory increase is under control

In [ ]:
memory = {}

In [ ]:
while True:

  # store memory consumption of children and timestamps
  process = psutil.Process()
  totalMem = 0
  for child in process.children(recursive=True):
    totalMem += child.memory_info().rss
    memory[child.pid] = memory.get(child.pid, []) + [(time.time(), child.memory_info().rss)]

  # plot memory vs time
  IPython.display.clear_output(True)
  figure(figsize=(14,5))
  xlabel('time since start of simulation (hours)')
  ylabel('reserved memory (GiB)')
  GbPerHours = []
  guiPid = None
  for pid, tmem in memory.items():
    if len(tmem) > 2:
      T, M = array(tmem).T
      plot((T-t0)/60**2, M/1024**3, label=f'{pid=}')

      # calculate and check/print memory increase slopes if enough data present
      if len(T) > 10:
        hi = 1
        while True:
          # calc for first hour in data
          _t, _m = T[T<min(T)+60*60], M[T<min(T)+60*60]
          # remove outliers from _m
          _filt = sorted(argsort(_m)[:-7])
          _t, _m = _t[_filt], _m[_filt]
          GbPerHour = polyfit(_t, _m, deg=1)[0]*60*60/1024**3
          GbPerHours.append(GbPerHour)
          print(f'hour {hi:2.0f} child {pid=:9.0f} memory increase: {GbPerHour*1024:9.1f} MiB/hour')

          # check maximum increase rate for gui worker is not exceeded
          if _t[-1]-_t[0] > 50*60 and GbPerHour > (5 if hi<=5 else 0.5):
            raise ValueError(f'memory increase rate of fastest growing process (probably GUI) is out of bounds: {GbPerHour=}')

          # allow above 0.01 only for one pid (for GUI process)
          if _t[-1]-_t[0] > 50*60 and GbPerHour > 0.01 and pid != guiPid:
            if guiPid is None:
              guiPid = pid
            else:
              raise ValueError(f'memory increase rate of background workers is out of bounds for at least one child: {GbPerHour=}')

          # shorten data
          T, M = T[T>=min(T)+60*60], M[T>=min(T)+60*60]
          hi += 1
          if len(T) < 10:
            break
  legend()
  show()

  # make sure total memory consumption is within limits
  if totalMem/1024**3 > 10:
    raise ValueError(f'memory increase rate of background workers is out of bounds for at least one child: {GbPerHour=}')

  # end of loop and limit loop speed
  if time.time()-t0 > 14*60*60:
    print(f'fourteen hours passed, exiting happily')
    break
  time.sleep(30)